# 03 - Crypto Multi-Pair Momentum Scan

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/huuan26/polars-anta/blob/master/examples/03_crypto_momentum_scan.ipynb)

In this notebook, we scan a basket of crypto pairs for **momentum breakouts** with **polars-anta**:
RSI, Bollinger Bands and relative volume are computed for every pair at once using `.over("pair")`, combined into a long signal, and backtested in a quick performance summary.

> The data is synthetic and the backtest ignores fees and slippage, so the results are illustrative only.

In [ ]:
# Install polars-anta (uncomment if running in Colab)
!pip install -q polars polars-anta matplotlib

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import polars as pl

import polars_anta  # Registers the .ta namespace

print(f"Polars Version: {pl.__version__}")
print(f"polars-anta Version: {polars_anta.__version__}")

## 1. Simulate OHLCV Bars for 8 Crypto Pairs

Volume rises with the size of each move, so volume spikes accompany breakouts. Swap in your own exchange data with the same columns (`pair`, `bar`, `open`, `high`, `low`, `close`, `volume`).

In [ ]:
pairs = ["BTC", "ETH", "SOL", "BNB", "XRP", "ADA", "AVAX", "DOGE"]
n_bars = 1000
np.random.seed(42)

ret = np.random.normal(0.0002, 0.01, (len(pairs), n_bars))
close = 100.0 * np.exp(np.cumsum(ret, axis=1))
open_ = np.concatenate([close[:, :1], close[:, :-1]], axis=1)
wick = np.abs(np.random.normal(0, 0.003, (2, *close.shape)))
high = np.maximum(open_, close) * (1.0 + wick[0])
low = np.minimum(open_, close) * (1.0 - wick[1])
volume = np.random.randint(1000, 5000, close.shape) * (1.0 + 50.0 * np.abs(ret))

df = pl.DataFrame(
    {
        "pair": np.repeat(pairs, n_bars),
        "bar": np.tile(np.arange(n_bars), len(pairs)),
        "open": open_.ravel(),
        "high": high.ravel(),
        "low": low.ravel(),
        "close": close.ravel(),
        "volume": volume.ravel(),
    }
)

print(f"{len(pairs)} pairs x {n_bars} bars = {df.height:,} rows")
print(df.head())

## 2. Compute Indicators for All Pairs with `.over('pair')`

RSI (14), Bollinger Bands (20, 2.0) and relative volume (volume / 20-bar average volume) in a single query:

In [ ]:
scan_df = df.with_columns(
    pl.col("close").ta.rsi(14).over("pair").alias("rsi_14"),
    pl.col("close").ta.bollinger_bands(20, 2.0).over("pair").alias("bb"),
    (pl.col("volume") / pl.col("volume").rolling_mean(20).over("pair")).alias("rvol"),
).unnest("bb")

print(scan_df.select(["pair", "bar", "close", "upper", "rsi_14", "rvol"]).tail())

## 3. Generate Breakout Signals

A pair gets a long signal on a bar when all three conditions hold:
1. **Breakout**: Close above the Upper Bollinger Band (`close > upper`)
2. **Momentum**: RSI between 55 and 80 (strong, but not overbought)
3. **Volume Confirmation**: Relative Volume $\ge 1.5$x the 20-bar average (`rvol >= 1.5`)

In [ ]:
signal_df = scan_df.with_columns(
    (
        (pl.col("close") > pl.col("upper"))
        & pl.col("rsi_14").is_between(55.0, 80.0)
        & (pl.col("rvol") >= 1.5)
    )
    .fill_null(False)
    .alias("long")
)

print(f"Latest signals ({signal_df['long'].sum():,} in total):")
print(signal_df.filter("long").select(["pair", "bar", "close", "upper", "rsi_14", "rvol"]).tail())

## 4. Backtest and Performance Summary

Each signal is entered at the close of its bar and held for the next bar (the signal is shifted by one bar to avoid look-ahead). Pairs are equal-weighted, and bars without a signal earn nothing.

In [ ]:
bt = signal_df.with_columns(
    pl.col("close").pct_change().over("pair").fill_null(0.0).alias("ret")
).with_columns(
    pl.when(pl.col("long").shift(1, fill_value=False).over("pair"))
    .then(pl.col("ret"))
    .otherwise(0.0)
    .alias("strategy_ret")
)

summary = bt.group_by("pair", maintain_order=True).agg(
    pl.col("long").sum().alias("signals"),
    ((1 + pl.col("strategy_ret")).product() - 1).alias("strategy_return"),
    ((1 + pl.col("ret")).product() - 1).alias("buy_hold_return"),
)
print(summary)

In [ ]:
equity = (
    bt.group_by("bar")
    .agg(
        pl.col("strategy_ret").mean().alias("strategy"),
        pl.col("ret").mean().alias("buy_hold"),
    )
    .sort("bar")
    .with_columns((pl.col("strategy", "buy_hold") + 1).cum_prod())
)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(equity["bar"], equity["strategy"], label="Momentum strategy", color="#1f77b4", lw=1.5)
ax.plot(equity["bar"], equity["buy_hold"], label="Buy & hold", color="#ff7f0e", lw=1.5)
ax.set_title(
    "Equal-Weight Portfolio: Momentum Strategy vs Buy & Hold", fontsize=13, fontweight="bold"
)
ax.set_xlabel("Bar")
ax.set_ylabel("Growth of 1.0")
ax.legend(loc="upper left")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()